[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/01-python/03_conditions.ipynb)

*Python from Scratch · Notebook 3 of 12*

# Conditions (if / elif / else)

Make decisions: if, elif, else, nested conditions, the one-line conditional and match-case.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** An if statement is a railway switch: the train takes exactly one track, and the first matching one wins.

**What interviewers really test here:** Order of conditions, edge cases at boundaries, and simplifying nested ifs.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. The if statement *(Basics)*
- [ ] 2. if / else and if / elif / else *(Basics)*
- [ ] 3. Combining conditions and nesting *(Core)*
- [ ] 4. One-line conditionals and truthiness *(Core)*
- [ ] 5. match-case (structural pattern matching) *(Advanced)*
- [ ] 6. Replacing long if-chains with data *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

**Control flow** is the order in which a program's statements run. By default it is **sequential** (top to bottom). **Conditional statements** (also called **selection** or **decision-making** statements) let the program choose which block to run based on a **condition** — an expression that evaluates to True or False.



The three basic control structures of structured programming are **sequence**, **selection** (if) and **iteration** (loops).

### Forms of the if statement

| Form | Syntax | Runs |
| --- | --- | --- |
| Simple if | `if cond:` block | block only when cond is truthy |
| if–else | `if cond:` A `else:` B | exactly one of A, B |
| if–elif–else ladder | `if c1:` A `elif c2:` B … `else:` Z | the first block whose condition is truthy, else Z |
| Nested if | an if inside another block | inner test only when the outer one passes |
| Conditional expression | `x if cond else y` | an **expression** that gives a value |
| match–case | `match value:` `case pattern:` | the first matching case |



A block can't be empty; use `pass` as a placeholder.

### How a condition is evaluated

1. The expression after `if`/`elif` is evaluated.
2. The result is converted to bool using **truthiness** (the same as `bool(value)`).
3. If True, the block runs and the rest of the ladder is skipped. If False, Python moves to the next `elif`, then `else`.



Falsy values: `False, None, 0, 0.0, "", [], (), {}, set(), range(0)`. Everything else is truthy, including `"0"`, `"False"` and `[0]`.



Because `and`/`or` short-circuit, parts of a condition may never run: `if x != 0 and 10 / x > 1` is safe.

### Indentation and blocks

Python uses **indentation** instead of braces to define a block. All statements in the same block must be indented by the same amount (PEP 8: 4 spaces, never mix tabs and spaces).



```python
if x > 0:
    print("positive")     # inside the if
    x -= 1                # inside the if
print("always")           # outside
```



Errors: `IndentationError: expected an indented block`, `unexpected indent`, and `SyntaxError: expected ':'`.

### match-case in detail

| Pattern | Example | Matches |
| --- | --- | --- |
| Literal | `case 404:` | the exact value |
| OR pattern | `case "y" \| "yes":` | any alternative |
| Capture | `case x:` | anything, binds it to x |
| Wildcard | `case _:` | anything (default) |
| Sequence | `case [first, *rest]:` | lists/tuples by shape |
| Mapping | `case {"type": "click", "x": x}:` | dicts with those keys |
| Class | `case Point(x=0, y=0):` | objects by type and attributes |
| Guard | `case n if n < 0:` | pattern plus an extra condition |



Unlike C's `switch`, there is no fall-through: only the first matching case runs.

### Good practices and common mistakes

- Order an elif ladder from the most restrictive condition to the least.
- Prefer `if x:` / `if not items:` over `if x == True:` / `if len(items) == 0:`.
- Compare with None using `is`: `if result is None:`.
- Avoid deep nesting — use guard clauses (early `return`) and combined conditions.
- Replace long value-to-value ladders with a dict or tuple lookup.
- Mistake: `if x = 5:` (assignment) instead of `if x == 5:` — a SyntaxError in Python.
- Mistake: `if x == 1 or 2:` is always true (it means `(x == 1) or 2`). Write `if x in (1, 2):`.

---
# Part 2: Lessons, step by step

## Lesson 1: The if statement

*Level: Basics*

Until now every line ran. An `if` statement runs a block of code **only when a condition is True**:



```python
temperature = 35
if temperature > 30:
    print("It's hot!")      # runs only if the condition is True
print("Done")               # always runs
```




- The condition is followed by a **colon** `:`.
- The lines that belong to the if are **indented** (4 spaces). That indented group is called a **block**.
- The first line that is not indented is outside the if.



**⚠️ Watch out:** IndentationError and "expected ':'" are the two most common beginner errors here.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def if_demo():
    temperature = 35
    if temperature > 30:
        print("It's hot!")
    temperature = 20
    if temperature > 30:
        print("This line is skipped")
    print("Done")


if_demo()

It's hot!
Done


**✏️ Your turn:** you now have what you need for [Q1](#q1). Try one now, or carry on and do them in Part 3.

## Lesson 2: if / else and if / elif / else

*Level: Basics*

`else` gives the block to run when the condition is False. `elif` ("else if") adds more choices:



```python
if marks >= 90:
    grade = "A"
elif marks >= 75:
    grade = "B"
else:
    grade = "C"
```



Python checks the conditions **from top to bottom** and runs only the **first** one that is True; the rest are skipped. That's why the order matters: put the strictest condition first.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def elif_demo():
    for marks in [95, 80, 52]:
        if marks >= 90:
            grade = "A"
        elif marks >= 75:
            grade = "B"
        else:
            grade = "C"
        print(marks, "->", grade)


elif_demo()

95 -> A
80 -> B
52 -> C


**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q3](#q3), [Q11](#q11), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 3: Combining conditions and nesting

*Level: Core*

Use `and`, `or`, `not` to combine conditions, or put one if inside another (**nesting**):



```python
if age >= 18 and has_id:          # combined
    print("Welcome")

if age >= 18:                     # nested
    if has_id:
        print("Welcome")
    else:
        print("Show your ID")
```



Combined conditions keep code flat; nesting is useful when the inner choice has its own else. More than two levels of nesting usually means you should restructure — e.g. return early.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def nested_demo():
    age = 20
    has_id = False
    if age >= 18:
        if has_id:
            print("Welcome in")
        else:
            print("Adult, but show your ID")
    else:
        print("Too young")
    can_enter = age >= 18 and has_id
    print("can_enter =", can_enter)


nested_demo()

Adult, but show your ID
can_enter = False


**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q14](#q14), [Q10](#q10), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 4: One-line conditionals and truthiness

*Level: Core*

A **conditional expression** picks one of two values in a single line:



```python
label = "even" if n % 2 == 0 else "odd"
```



Read it as: *value\_if\_true* **if** *condition* **else** *value\_if\_false*.



A condition doesn't have to be a comparison. Any value works: empty strings, empty lists, 0 and None count as False; everything else counts as True.



```python
name = ""
if not name:
    print("Please enter a name")
```

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [5]:
def ternary_demo():
    for n in [4, 7]:
        label = "even" if n % 2 == 0 else "odd"
        print(n, label)
    cart = []
    if not cart:
        print("Your cart is empty")
    cart.append("pen")
    if cart:
        print("Items:", len(cart))


ternary_demo()

4 even
7 odd
Your cart is empty
Items: 1


**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q5](#q5). Try one now, or carry on and do them in Part 3.

## Lesson 5: match-case (structural pattern matching)

*Level: Advanced*

Python 3.10 added `match`, which compares one value against several **patterns**:



```python
match command:
    case "start":
        ...
    case "stop" | "quit":        # either value
        ...
    case ("move", x, y):         # a tuple; x and y are captured
        ...
    case _:                      # anything else (default)
        ...
```



Cases can have **guards**: `case "/" if b != 0:`. Patterns can also match lists, dicts and objects. For a few simple equality checks, if/elif is just as good.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [6]:
def match_demo():
    def describe(cmd):
        match cmd:
            case "start":
                return "starting"
            case "stop" | "quit":
                return "stopping"
            case ("move", x, y):
                return f"moving to {x},{y}"
            case _:
                return "unknown"

    for c in ["start", "quit", ("move", 3, 4), "jump"]:
        print(describe(c))


match_demo()

starting
stopping
moving to 3,4
unknown


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. describe() returns 'starting'
2. describe() returns 'stopping'
3. describe() returns 'moving to 3,4'
4. describe() returns 'unknown'
5. match\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q15](#q15). Try one now, or carry on and do them in Part 3.

## Lesson 6: Replacing long if-chains with data

*Level: Advanced*

When every branch just maps an input to an output, a **lookup table** (tuple, list or dict) is shorter, faster to change and less error-prone than many elif lines:



```python
days = ("Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun")
name = days[n - 1]

beats = {"rock": "scissors", "scissors": "paper", "paper": "rock"}
winner = "p1" if beats[p1] == p2 else "p2"
```



Another clean style is the **guard clause**: handle the special or invalid case first and `return` immediately, so the main logic isn't nested.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [7]:
def lookup_demo():
    def tier_discount(amount):
        for threshold, percent in [(5000, 20), (2000, 10), (1000, 5)]:
            if amount >= threshold:
                return percent
        return 0

    print(tier_discount(2500), tier_discount(300))


lookup_demo()

10 0


<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. tier\_discount() returns 10
2. tier\_discount() returns 0
3. lookup\_demo() returns None

</details>


**✏️ Your turn:** you now have what you need for [Q6](#q6), [Q17](#q17), [Q13](#q13), [Q9](#q9), [Q16](#q16). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Positive, Negative or Zero

🟢 Warm-up · *if / elif / else*

Return `'positive'`, `'negative'` or `'zero'` for the number `n`.

**Examples**

- `n = 5` → `'positive'`
- `n = -2` → `'negative'`
- `n = 0` → `'zero'`

In [ ]:
def sign_word(n):
    """
    Return 'positive', 'negative' or 'zero' for the number n.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sign_word, [
    ((5,), 'positive'),
    ((-2,), 'negative'),
    ((0,), 'zero'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Three separate if statements, each with its own return.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One if/elif/else chain — exactly one branch runs.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Three separate if statements, each with its own return.

```python
def sign_word(n):
    if n > 0:
        return "positive"
    if n < 0:
        return "negative"
    if n == 0:
        return "zero"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** One if/elif/else chain — exactly one branch runs.

```python
def sign_word(n):
    if n > 0:
        return "positive"
    elif n < 0:
        return "negative"
    else:
        return "zero"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Three separate if statements, each with its own return. That costs O(1) time and O(1) space."

"I can do better: One if/elif/else chain — exactly one branch runs. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/positive-negative-or-zero)

<a id="q2"></a>
### Q2. Grade From Marks

🟢 Warm-up · *elif chain*

Marks 90+ → 'A', 75–89 → 'B', 60–74 → 'C', 40–59 → 'D', below 40 → 'F'.

**Examples**

- `marks = 95` → `'A'`
- `marks = 75` → `'B'`
- `marks = 61` → `'C'`

In [ ]:
def grade(marks):
    """
    Marks 90+ → 'A', 75–89 → 'B', 60–74 → 'C', 40–59 → 'D', below 40 → 'F'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(grade, [
    ((95,), 'A'),
    ((75,), 'B'),
    ((61,), 'C'),
    ((40,), 'D'),
    ((12,), 'F'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Write each band with both lower and upper limits.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Check from the top down: each elif already knows the higher bands failed, so only lower limits are needed. A table of cut-offs makes it data-driven.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Write each band with both lower and upper limits.

```python
def grade(marks):
    if marks >= 90:
        return "A"
    if marks >= 75 and marks < 90:
        return "B"
    if marks >= 60 and marks < 75:
        return "C"
    if marks >= 40 and marks < 60:
        return "D"
    return "F"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Check from the top down: each elif already knows the higher bands failed, so only lower limits are needed. A table of cut-offs makes it data-driven.

```python
def grade(marks):
    for cutoff, letter in [(90, "A"), (75, "B"), (60, "C"), (40, "D")]:
        if marks >= cutoff:
            return letter
    return "F"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Write each band with both lower and upper limits. That costs O(1) time and O(1) space."

"I can do better: Check from the top down: each elif already knows the higher bands failed, so only lower limits are needed. A table of cut-offs makes it data-driven. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/grade-from-marks)

<a id="q3"></a>
### Q3. Ticket Price by Age

🟢 Warm-up · *Ranges*

Under 5 → 0, 5–17 → 50, 18–59 → 100, 60 and above → 60. Return the price.

**Examples**

- `age = 3` → `0`
- `age = 12` → `50`
- `age = 30` → `100`

In [ ]:
def ticket(age):
    """
    Under 5 → 0, 5–17 → 50, 18–59 → 100, 60 and above → 60. Return the price.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ticket, [
    ((3,), 0),
    ((12,), 50),
    ((30,), 100),
    ((65,), 60),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Separate if statements, each checking a full range with `and`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

An elif chain in increasing order needs only one comparison per branch.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Separate if statements, each checking a full range with `and`.

```python
def ticket(age):
    price = 0
    if age < 5:
        price = 0
    if age >= 5 and age <= 17:
        price = 50
    if age >= 18 and age <= 59:
        price = 100
    if age >= 60:
        price = 60
    return price
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** An elif chain in increasing order needs only one comparison per branch.

```python
def ticket(age):
    if age < 5:
        return 0
    elif age < 18:
        return 50
    elif age < 60:
        return 100
    return 60
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Separate if statements, each checking a full range with `and`. That costs O(1) time and O(1) space."

"I can do better: An elif chain in increasing order needs only one comparison per branch. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/ticket-price-by-age)

<a id="q4"></a>
### Q4. Larger of Two

🟢 Warm-up · *Conditional expression*

Return the larger of `a` and `b`.

**Examples**

- `a = 3, b = 8` → `8`
- `a = 9, b = 2` → `9`
- `a = 4, b = 4` → `4`

In [ ]:
def larger(a, b):
    """
    Return the larger of a and b.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(larger, [
    ((3, 8), 8),
    ((9, 2), 9),
    ((4, 4), 4),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A full if/else statement.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The one-line conditional expression `a if a >= b else b` (or simply `max`).

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** A full if/else statement.

```python
def larger(a, b):
    if a >= b:
        result = a
    else:
        result = b
    return result
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** The one-line conditional expression `a if a >= b else b` (or simply `max`).

```python
def larger(a, b):
    return a if a >= b else b
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A full if/else statement. That costs O(1) time and O(1) space."

"I can do better: The one-line conditional expression `a if a >= b else b` (or simply `max`). That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/larger-of-two)

<a id="q5"></a>
### Q5. Vowel or Consonant

🟢 Warm-up · *in operator*

For a single letter, return `'vowel'` or `'consonant'` (case doesn't matter).

**Examples**

- `ch = 'a'` → `'vowel'`
- `ch = 'E'` → `'vowel'`
- `ch = 'z'` → `'consonant'`

In [ ]:
def vowel_or_consonant(ch):
    """
    For a single letter, return 'vowel' or 'consonant' (case doesn't matter).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(vowel_or_consonant, [
    (('a',), 'vowel'),
    (('E',), 'vowel'),
    (('z',), 'consonant'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare with each vowel using `or`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Lower-case it and test membership: `ch in 'aeiou'`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Compare with each vowel using `or`.

```python
def vowel_or_consonant(ch):
    ch = ch.lower()
    if ch == "a" or ch == "e" or ch == "i" or ch == "o" or ch == "u":
        return "vowel"
    return "consonant"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Lower-case it and test membership: `ch in 'aeiou'`.

```python
def vowel_or_consonant(ch):
    return "vowel" if ch.lower() in "aeiou" else "consonant"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare with each vowel using `or`. That costs O(1) time and O(1) space."

"I can do better: Lower-case it and test membership: `ch in 'aeiou'`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/vowel-or-consonant)

<a id="q6"></a>
### Q6. Day Name

🟢 Warm-up · *elif chain vs lookup*

Return the day name for numbers 1–7 (1 = 'Monday'); otherwise return `'invalid'`.

**Examples**

- `n = 1` → `'Monday'`
- `n = 7` → `'Sunday'`
- `n = 9` → `'invalid'`

In [ ]:
def day_name(n):
    """
    Return the day name for numbers 1–7 (1 = 'Monday'); otherwise return
    'invalid'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(day_name, [
    ((1,), 'Monday'),
    ((7,), 'Sunday'),
    ((9,), 'invalid'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Seven if/elif branches.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Store the names in a tuple and index it — a lookup replaces the whole chain.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Seven if/elif branches.

```python
def day_name(n):
    if n == 1:
        return "Monday"
    elif n == 2:
        return "Tuesday"
    elif n == 3:
        return "Wednesday"
    elif n == 4:
        return "Thursday"
    elif n == 5:
        return "Friday"
    elif n == 6:
        return "Saturday"
    elif n == 7:
        return "Sunday"
    return "invalid"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Store the names in a tuple and index it — a lookup replaces the whole chain.

```python
def day_name(n):
    days = ("Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday")
    return days[n - 1] if 1 <= n <= 7 else "invalid"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Seven if/elif branches. That costs O(1) time and O(1) space."

"I can do better: Store the names in a tuple and index it — a lookup replaces the whole chain. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/day-name)

<a id="q7"></a>
### Q7. BMI Category

🟢 Warm-up · *Float conditions*

BMI = weight / height². Below 18.5 → 'under', below 25 → 'normal', below 30 → 'over', else 'obese'.

**Examples**

- `weight = 50, height = 1.8` → `'under'`
- `weight = 70, height = 1.75` → `'normal'`
- `weight = 85, height = 1.75` → `'over'`

In [ ]:
def bmi_category(weight, height):
    """
    BMI = weight / height². Below 18.5 → 'under', below 25 → 'normal', below 30
    → 'over', else 'obese'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(bmi_category, [
    ((50, 1.8), 'under'),
    ((70, 1.75), 'normal'),
    ((85, 1.75), 'over'),
    ((110, 1.7), 'obese'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute BMI and test full ranges.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Ordered elif chain with only upper limits.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Compute BMI and test full ranges.

```python
def bmi_category(weight, height):
    bmi = weight / (height * height)
    if bmi < 18.5:
        return "under"
    if bmi >= 18.5 and bmi < 25:
        return "normal"
    if bmi >= 25 and bmi < 30:
        return "over"
    return "obese"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Ordered elif chain with only upper limits.

```python
def bmi_category(weight, height):
    bmi = weight / height ** 2
    if bmi < 18.5:
        return "under"
    elif bmi < 25:
        return "normal"
    elif bmi < 30:
        return "over"
    return "obese"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute BMI and test full ranges. That costs O(1) time and O(1) space."

"I can do better: Ordered elif chain with only upper limits. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/bmi-category)

<a id="q8"></a>
### Q8. Quadrant of a Point

🟢 Warm-up · *Nested conditions*

Return 1, 2, 3 or 4 for the quadrant of (x, y), or 0 if it lies on an axis.

**Examples**

- `x = 3, y = 4` → `1`
- `x = -3, y = 4` → `2`
- `x = -1, y = -1` → `3`

In [ ]:
def quadrant(x, y):
    """
    Return 1, 2, 3 or 4 for the quadrant of (x, y), or 0 if it lies on an axis.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(quadrant, [
    ((3, 4), 1),
    ((-3, 4), 2),
    ((-1, -1), 3),
    ((2, -5), 4),
    ((0, 7), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested if: first on x, then on y.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Handle the axis case first (guard clause), then one flat chain using `and`.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Nested if: first on x, then on y.

```python
def quadrant(x, y):
    if x > 0:
        if y > 0:
            return 1
        elif y < 0:
            return 4
        else:
            return 0
    elif x < 0:
        if y > 0:
            return 2
        elif y < 0:
            return 3
        else:
            return 0
    else:
        return 0
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Handle the axis case first (guard clause), then one flat chain using `and`.

```python
def quadrant(x, y):
    if x == 0 or y == 0:
        return 0
    if x > 0:
        return 1 if y > 0 else 4
    return 2 if y > 0 else 3
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested if: first on x, then on y. That costs O(1) time and O(1) space."

"I can do better: Handle the axis case first (guard clause), then one flat chain using `and`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/quadrant-of-a-point)

<a id="q9"></a>
### Q9. Shopping Discount

🟢 Warm-up · *Tiers*

Purchases of 5000+ get 20% off, 2000+ get 10% off, 1000+ get 5% off. Return the final amount.

**Examples**

- `amount = 6000` → `4800.0`
- `amount = 2500` → `2250.0`
- `amount = 1000` → `950.0`

In [ ]:
def discounted(amount):
    """
    Purchases of 5000+ get 20% off, 2000+ get 10% off, 1000+ get 5% off. Return
    the final amount.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(discounted, [
    ((6000,), 4800.0),
    ((2500,), 2250.0),
    ((1000,), 950.0),
    ((500,), 500.0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute the discount percent with ifs, then apply it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Walk a list of (threshold, percent) tiers from the top.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Compute the discount percent with ifs, then apply it.

```python
def discounted(amount):
    percent = 0
    if amount >= 1000:
        percent = 5
    if amount >= 2000:
        percent = 10
    if amount >= 5000:
        percent = 20
    return amount - amount * percent / 100
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Walk a list of (threshold, percent) tiers from the top.

```python
def discounted(amount):
    for threshold, percent in [(5000, 20), (2000, 10), (1000, 5)]:
        if amount >= threshold:
            return amount * (1 - percent / 100)
    return float(amount)
```

**Cost:** O(tiers) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute the discount percent with ifs, then apply it. That costs O(1) time and O(1) space."

"I can do better: Walk a list of (threshold, percent) tiers from the top. That's O(tiers) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/shopping-discount)

<a id="q10"></a>
### Q10. Can It Be a Triangle?

🟢 Warm-up · *and*

Return True if three positive lengths can form a triangle (each pair sums to more than the third).

**Examples**

- `a = 3, b = 4, c = 5` → `True`
- `a = 1, b = 1, c = 3` → `False`
- `a = 5, b = 5, c = 5` → `True`

In [ ]:
def can_be_triangle(a, b, c):
    """
    Return True if three positive lengths can form a triangle (each pair sums to
    more than the third).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(can_be_triangle, [
    ((3, 4, 5), True),
    ((1, 1, 3), False),
    ((5, 5, 5), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Three ifs, returning False on any failure.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Only the largest side can break the rule: sum of all &gt; 2 × max.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Three ifs, returning False on any failure.

```python
def can_be_triangle(a, b, c):
    if a + b <= c:
        return False
    if a + c <= b:
        return False
    if b + c <= a:
        return False
    return True
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Only the largest side can break the rule: sum of all &gt; 2 × max.

```python
def can_be_triangle(a, b, c):
    return a + b + c > 2 * max(a, b, c)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Three ifs, returning False on any failure. That costs O(1) time and O(1) space."

"I can do better: Only the largest side can break the rule: sum of all &gt; 2 × max. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/can-it-be-a-triangle)

<a id="q11"></a>
### Q11. Greeting by Hour

🟢 Warm-up · *Chained comparisons*

For an hour 0–23 return 'morning' (5–11), 'afternoon' (12–16), 'evening' (17–20) or 'night'.

**Examples**

- `hour = 8` → `'morning'`
- `hour = 13` → `'afternoon'`
- `hour = 19` → `'evening'`

In [ ]:
def greeting(hour):
    """
    For an hour 0–23 return 'morning' (5–11), 'afternoon' (12–16), 'evening'
    (17–20) or 'night'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(greeting, [
    ((8,), 'morning'),
    ((13,), 'afternoon'),
    ((19,), 'evening'),
    ((2,), 'night'),
    ((23,), 'night'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Ranges written with `and`.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Chained comparisons read like the specification.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Ranges written with `and`.

```python
def greeting(hour):
    if hour >= 5 and hour <= 11:
        return "morning"
    if hour >= 12 and hour <= 16:
        return "afternoon"
    if hour >= 17 and hour <= 20:
        return "evening"
    return "night"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Chained comparisons read like the specification.

```python
def greeting(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 21:
        return "evening"
    return "night"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Ranges written with `and`. That costs O(1) time and O(1) space."

"I can do better: Chained comparisons read like the specification. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/greeting-by-hour)

<a id="q12"></a>
### Q12. Character Type

🟢 Warm-up · *str methods in conditions*

Return 'upper', 'lower', 'digit', 'space' or 'other' for a single character.

**Examples**

- `ch = 'A'` → `'upper'`
- `ch = 'q'` → `'lower'`
- `ch = '7'` → `'digit'`

In [ ]:
def char_type(ch):
    """
    Return 'upper', 'lower', 'digit', 'space' or 'other' for a single character.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(char_type, [
    (('A',), 'upper'),
    (('q',), 'lower'),
    (('7',), 'digit'),
    ((' ',), 'space'),
    (('#',), 'other'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare against character ranges like 'A' &lt;= ch &lt;= 'Z'.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Use the string methods isupper / islower / isdigit / isspace.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Compare against character ranges like 'A' &lt;= ch &lt;= 'Z'.

```python
def char_type(ch):
    if "A" <= ch <= "Z":
        return "upper"
    if "a" <= ch <= "z":
        return "lower"
    if "0" <= ch <= "9":
        return "digit"
    if ch == " " or ch == "\t" or ch == "\n":
        return "space"
    return "other"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Use the string methods isupper / islower / isdigit / isspace.

```python
def char_type(ch):
    if ch.isupper():
        return "upper"
    elif ch.islower():
        return "lower"
    elif ch.isdigit():
        return "digit"
    elif ch.isspace():
        return "space"
    return "other"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare against character ranges like 'A' &lt;= ch &lt;= 'Z'. That costs O(1) time and O(1) space."

"I can do better: Use the string methods isupper / islower / isdigit / isspace. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/character-type)

<a id="q13"></a>
### Q13. Next Traffic Light

🟢 Warm-up · *Lookup instead of if*

Return the next colour: green → yellow → red → green.

**Examples**

- `colour = 'green'` → `'yellow'`
- `colour = 'yellow'` → `'red'`
- `colour = 'red'` → `'green'`

In [ ]:
def next_light(colour):
    """
    Return the next colour: green → yellow → red → green.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(next_light, [
    (('green',), 'yellow'),
    (('yellow',), 'red'),
    (('red',), 'green'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if/elif for each colour.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dict maps each colour to the next one.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** An if/elif for each colour.

```python
def next_light(colour):
    if colour == "green":
        return "yellow"
    elif colour == "yellow":
        return "red"
    else:
        return "green"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A dict maps each colour to the next one.

```python
def next_light(colour):
    return {"green": "yellow", "yellow": "red", "red": "green"}[colour]
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if/elif for each colour. That costs O(1) time and O(1) space."

"I can do better: A dict maps each colour to the next one. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/next-traffic-light)

<a id="q14"></a>
### Q14. Triangle Type

🟡 Interview standard · *Combined conditions*

Given three side lengths, return `'invalid'` if they can't form a triangle, else `'equilateral'`, `'isosceles'` or `'scalene'`.

**Examples**

- `a = 3, b = 3, c = 3` → `'equilateral'`
- `a = 3, b = 3, c = 5` → `'isosceles'`
- `a = 3, b = 4, c = 5` → `'scalene'`

In [ ]:
def triangle(a, b, c):
    """
    Given three side lengths, return 'invalid' if they can't form a triangle,
    else 'equilateral', 'isosceles' or 'scalene'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(triangle, [
    ((3, 3, 3), 'equilateral'),
    ((3, 3, 5), 'isosceles'),
    ((3, 4, 5), 'scalene'),
    ((1, 2, 3), 'invalid'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check validity with three separate comparisons, then count equal pairs with ifs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort the sides so validity is one test (a + b &gt; c), then the number of distinct sides tells the type.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check validity with three separate comparisons, then count equal pairs with ifs.

```python
def triangle(a, b, c):
    if a + b <= c:
        return "invalid"
    if a + c <= b:
        return "invalid"
    if b + c <= a:
        return "invalid"
    if a == b and b == c:
        return "equilateral"
    if a == b or b == c or a == c:
        return "isosceles"
    return "scalene"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Sort the sides so validity is one test (a + b &gt; c), then the number of distinct sides tells the type.

```python
def triangle(a, b, c):
    x, y, z = sorted((a, b, c))
    if x + y <= z:
        return "invalid"
    return {1: "equilateral", 2: "isosceles", 3: "scalene"}[len({a, b, c})]
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check validity with three separate comparisons, then count equal pairs with ifs. That costs O(1) time and O(1) space."

"I can do better: Sort the sides so validity is one test (a + b &gt; c), then the number of distinct sides tells the type. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/triangle-type)

<a id="q15"></a>
### Q15. Simple Calculator

🟡 Interview standard · *match-case*

Given `a`, an operator string (`'+'`, `'-'`, `'*'`, `'/'`) and `b`, return the result, or `None` for an unknown operator or division by zero.

**Examples**

- `a = 6, op = '+', b = 3` → `9`
- `a = 6, op = '/', b = 3` → `2.0`
- `a = 6, op = '/', b = 0` → `None`

In [ ]:
def calc(a, op, b):
    """
    Given a, an operator string ('+', '-', '*', '/') and b, return the result,
    or None for an unknown operator or division by zero.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(calc, [
    ((6, '+', 3), 9),
    ((6, '/', 3), 2.0),
    ((6, '/', 0), None),
    ((6, '%', 3), None),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if/elif chain comparing the operator.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A `match` statement reads like a table of cases; a guard (`if b != 0`) handles division by zero.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** An if/elif chain comparing the operator.

```python
def calc(a, op, b):
    if op == "+":
        return a + b
    elif op == "-":
        return a - b
    elif op == "*":
        return a * b
    elif op == "/":
        if b == 0:
            return None
        return a / b
    else:
        return None
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A `match` statement reads like a table of cases; a guard (`if b != 0`) handles division by zero.

```python
def calc(a, op, b):
    match op:
        case "+":
            return a + b
        case "-":
            return a - b
        case "*":
            return a * b
        case "/" if b != 0:
            return a / b
        case _:
            return None
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if/elif chain comparing the operator. That costs O(1) time and O(1) space."

"I can do better: A `match` statement reads like a table of cases; a guard (`if b != 0`) handles division by zero. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/simple-calculator)

<a id="q16"></a>
### Q16. Electricity Bill

🟡 Interview standard · *Slabs*

First 100 units cost 5 each, the next 100 cost 7 each, anything above 200 costs 10 each. Return the bill.

**Examples**

- `units = 50` → `250`
- `units = 150` → `850`
- `units = 250` → `1700`

In [ ]:
def bill(units):
    """
    First 100 units cost 5 each, the next 100 cost 7 each, anything above 200
    costs 10 each. Return the bill.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(bill, [
    ((50,), 250),
    ((150,), 850),
    ((250,), 1700),
    ((0,), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Branch on which slab the total falls into and write the full sum for each case.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Charge each slab for the units that fall inside it with min/max — no special cases.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Branch on which slab the total falls into and write the full sum for each case.

```python
def bill(units):
    if units <= 100:
        return units * 5
    elif units <= 200:
        return 100 * 5 + (units - 100) * 7
    else:
        return 100 * 5 + 100 * 7 + (units - 200) * 10
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Charge each slab for the units that fall inside it with min/max — no special cases.

```python
def bill(units):
    total = 0
    for start, end, price in [(0, 100, 5), (100, 200, 7), (200, float("inf"), 10)]:
        if units > start:
            total += (min(units, end) - start) * price
    return int(total)
```

**Cost:** O(slabs) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Branch on which slab the total falls into and write the full sum for each case. That costs O(1) time and O(1) space."

"I can do better: Charge each slab for the units that fall inside it with min/max — no special cases. That's O(slabs) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/electricity-bill)

<a id="q17"></a>
### Q17. Rock Paper Scissors

🟡 Interview standard · *Conditions vs lookup*

Return `'p1'`, `'p2'` or `'draw'` for moves 'rock', 'paper', 'scissors'.

**Examples**

- `p1 = 'rock', p2 = 'scissors'` → `'p1'`
- `p1 = 'rock', p2 = 'paper'` → `'p2'`
- `p1 = 'paper', p2 = 'paper'` → `'draw'`

In [ ]:
def rps(p1, p2):
    """
    Return 'p1', 'p2' or 'draw' for moves 'rock', 'paper', 'scissors'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rps, [
    (('rock', 'scissors'), 'p1'),
    (('rock', 'paper'), 'p2'),
    (('paper', 'paper'), 'draw'),
    (('scissors', 'paper'), 'p1'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Check every winning combination for player 1 with if statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Store what each move beats in a dict; then the logic is two lines.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Check every winning combination for player 1 with if statements.

```python
def rps(p1, p2):
    if p1 == p2:
        return "draw"
    if p1 == "rock" and p2 == "scissors":
        return "p1"
    if p1 == "scissors" and p2 == "paper":
        return "p1"
    if p1 == "paper" and p2 == "rock":
        return "p1"
    return "p2"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** Store what each move beats in a dict; then the logic is two lines.

```python
def rps(p1, p2):
    beats = {"rock": "scissors", "scissors": "paper", "paper": "rock"}
    if p1 == p2:
        return "draw"
    return "p1" if beats[p1] == p2 else "p2"
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Check every winning combination for player 1 with if statements. That costs O(1) time and O(1) space."

"I can do better: Store what each move beats in a dict; then the logic is two lines. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/rock-paper-scissors)

<a id="q18"></a>
### Q18. Sort Three Numbers

🟡 Interview standard · *Swaps with if*

Return `(a, b, c)` in increasing order.

**Examples**

- `a = 3, b = 1, c = 2` → `(1, 2, 3)`
- `a = 5, b = 5, c = 1` → `(1, 5, 5)`
- `a = 1, b = 2, c = 3` → `(1, 2, 3)`

In [ ]:
def sort3(a, b, c):
    """
    Return (a, b, c) in increasing order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sort3, [
    ((3, 1, 2), (1, 2, 3)),
    ((5, 5, 1), (1, 5, 5)),
    ((1, 2, 3), (1, 2, 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Three compare-and-swap steps (a tiny sorting network).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`sorted()` returns a sorted list; convert it to a tuple.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Three compare-and-swap steps (a tiny sorting network).

```python
def sort3(a, b, c):
    if a > b:
        a, b = b, a
    if b > c:
        b, c = c, b
    if a > b:
        a, b = b, a
    return a, b, c
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `sorted()` returns a sorted list; convert it to a tuple.

```python
def sort3(a, b, c):
    return tuple(sorted((a, b, c)))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Three compare-and-swap steps (a tiny sorting network). That costs O(1) time and O(1) space."

"I can do better: `sorted()` returns a sorted list; convert it to a tuple. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/sort-three-numbers)

<a id="q19"></a>
### Q19. Strong Password

🟡 Interview standard · *Conditions with any()*

A password is strong if it has at least 8 characters, a digit, an uppercase letter and a lowercase letter. Return True or False.

**Examples**

- `password = 'Secret123'` → `True`
- `password = 'secret123'` → `False`
- `password = 'Sh0rt'` → `False`

In [ ]:
def strong(password):
    """
    A password is strong if it has at least 8 characters, a digit, an uppercase
    letter and a lowercase letter. Return True or False.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(strong, [
    (('Secret123',), True),
    (('secret123',), False),
    (('Sh0rt',), False),
    (('NoDigitsHere',), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop once, setting a flag for each kind of character, then check all flags.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`any()` with a generator tests 'is there at least one…' in one expression each.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Loop once, setting a flag for each kind of character, then check all flags.

```python
def strong(password):
    has_digit = False
    has_upper = False
    has_lower = False
    for ch in password:
        if ch.isdigit():
            has_digit = True
        if ch.isupper():
            has_upper = True
        if ch.islower():
            has_lower = True
    if len(password) >= 8 and has_digit and has_upper and has_lower:
        return True
    return False
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** `any()` with a generator tests 'is there at least one…' in one expression each.

```python
def strong(password):
    return (len(password) >= 8
            and any(c.isdigit() for c in password)
            and any(c.isupper() for c in password)
            and any(c.islower() for c in password))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop once, setting a flag for each kind of character, then check all flags. That costs O(n) time and O(1) space."

"I can do better: `any()` with a generator tests 'is there at least one…' in one expression each. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `password = 'Ab3cdefg'`

1. GeneratorExit raised: 
2. GeneratorExit raised: 
3. GeneratorExit raised: 
4. strong() returns True

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/strong-password)

<a id="q20"></a>
### Q20. Days in a Month

🟡 Interview standard · *Conditions + lookup*

Return the number of days in `month` (1–12) of `year`, handling leap years.

**Examples**

- `month = 2, year = 2024` → `29`
- `month = 2, year = 2023` → `28`
- `month = 4, year = 2023` → `30`

In [ ]:
def days_in_month(month, year):
    """
    Return the number of days in month (1–12) of year, handling leap years.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(days_in_month, [
    ((2, 2024), 29),
    ((2, 2023), 28),
    ((4, 2023), 30),
    ((12, 2023), 31),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Group the months in if/elif and test leap-year rules for February.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A tuple of month lengths plus one leap-year adjustment.

</details>

<details>
<summary><b>✅ Answer: Brute force — the long way</b></summary>

**Idea:** Group the months in if/elif and test leap-year rules for February.

```python
def days_in_month(month, year):
    if month == 2:
        if (year % 4 == 0 and year % 100 != 0) or year % 400 == 0:
            return 29
        return 28
    elif month == 4 or month == 6 or month == 9 or month == 11:
        return 30
    else:
        return 31
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: Optimal — the Pythonic way</b></summary>

**Idea:** A tuple of month lengths plus one leap-year adjustment.

```python
def days_in_month(month, year):
    leap = year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)
    days = (31, 29 if leap else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31)
    return days[month - 1]
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Group the months in if/elif and test leap-year rules for February. That costs O(1) time and O(1) space."

"I can do better: A tuple of month lengths plus one leap-year adjustment. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/python-course/#/q/conditions/days-in-a-month)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What is the difference between if–elif and multiple ifs?</b></summary>

In an if–elif ladder at most one block runs: once a condition is true the rest are skipped. With separate ifs every condition is checked independently and several blocks can run.

</details>

<details>
<summary><b>2. Does Python have a switch statement?</b></summary>

Since Python 3.10 it has `match`–`case`, which is more powerful than a classic switch (it matches structure and captures values). Before that, people used if–elif ladders or dict lookups.

</details>

<details>
<summary><b>3. What is a ternary operator in Python?</b></summary>

The conditional expression `a if condition else b`. It evaluates to a when the condition is true and b otherwise, and can be used inside other expressions.

</details>

<details>
<summary><b>4. Why is if x == 1 or 2: always true?</b></summary>

It is parsed as `(x == 1) or 2`, and 2 is truthy. Use `x == 1 or x == 2` or `x in (1, 2)`.

</details>

<details>
<summary><b>5. What does pass do?</b></summary>

Nothing. It's a placeholder where Python requires a statement but you have nothing to run yet, such as an empty if block.

</details>

<details>
<summary><b>6. Which values are falsy?</b></summary>

`False`, `None`, numeric zero (`0`, `0.0`, `0j`), and empty collections (`''`, `[]`, `()`, `{}`, `set()`, `range(0)`). Objects can define their own truthiness with `__bool__` or `__len__`.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: The if statement
- [ ] I can explain: if / else and if / elif / else
- [ ] I can explain: Combining conditions and nesting
- [ ] I can explain: One-line conditionals and truthiness
- [ ] I can explain: match-case (structural pattern matching)
- [ ] I can explain: Replacing long if-chains with data

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Electricity Bill
- [ ] [Q17](#q17) Rock Paper Scissors
- [ ] [Q18](#q18) Sort Three Numbers
- [ ] [Q19](#q19) Strong Password
- [ ] [Q20](#q20) Days in a Month

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.